# Qiskit Fall Fest 2026, Session 1: Quantum and Qiskit 101

Mon, Nov 16, 2026, 5:00 to 6:00 PM, on Zoom

Covers: Qubits, gates, entanglement, and the four steps of a Qiskit program.

This is the first of five sessions this week. Each session has a seminar, then a lab, then a challenge. Nobody is expected to know quantum physics. Everybody is expected to run code.

The words in the line above, in one sentence each. A qubit is the quantum version of a bit. A gate is one operation applied to qubits. Entanglement is what you have when several qubits share one state that no single qubit holds on its own. Qiskit is the open-source Python library, developed by IBM, that you use all week to write quantum programs and run them. Part 1 builds each idea from the start, so none of them is assumed.

## What you will be able to do

- Describe a qubit by two numbers, its amplitudes, and turn amplitudes into measurement probabilities.
- Read and build small circuits from the gates X, Z, H and CX. A circuit is a list of gates applied to qubits, usually ending in a measurement.
- Create a Bell pair and a GHZ state, two entangled states, and say what their measurement counts (how often each result comes up) should look like.
- Run a circuit through the four steps of a Qiskit program, on a simulator (a program that imitates a quantum computer) and on a real IBM quantum processor (the chip that holds the qubits).

## How to use this notebook

- Run the cells in order, from the top. A cell is one box of text or code. To run a code cell, click it and press Shift+Enter.
- Part 1 follows tonight's slides. Each idea has a short recap, and often a small cell you can run to see the idea at work.
- Part 2 is the lab. The code under "Try it" and "Lab" is complete. Run it, read the output, then change it and run it again.
- After each lab there is a "Your turn" exercise. Replace the line `# YOUR CODE HERE` with your own code, run the cell, then run the check cell below it. A check that fails prints a hint.
- Everything runs on a simulator, on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser), unless you change the switches in the Setup section.
- Use this notebook during the session and again afterwards on your own.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## Join the Discord and register

*Slide 2.*

- **Discord.** Join the server at https://discord.gg/FBNzRcxMr. Questions between sessions go there.
- **Registration.** Not registered yet? Fill in the form at https://forms.gle/NC9o1iRk3WggLuTH7.

On the slide the two links are QR codes, small square patterns that a phone camera reads as a link. Discord is the chat service where the club answers questions.

## Accounts

You need an account only to run on a real quantum computer. Everything in this notebook runs without one, on the simulator. Set the accounts up now, so that your first hardware job works when you send it. A job is one request to run circuits, sent to a quantum computer or to a simulator.

This section follows the setup slides. IBM Quantum Platform comes first. The second option comes after it.

### Two accounts before we start

*Slide 4.*

- **IBM Quantum.** quantum.cloud.ibm.com. Open Plan, free. Your circuits run here, on IBM processors, with your own API key.
- **qBraid, the second option.** account.qbraid.com. Sign up with the email you registered with. It reaches other quantum computers too.

Use IBM Quantum first. qBraid access is sent to the email on your registration form.

Two terms from the first card. An API key is a secret string of characters that lets a program, such as this notebook, act for your account. A plan is the kind of access an account has, and the Open Plan is the one you do not pay for.

Jobs on IBM processors always go through your own IBM Quantum account. The second option does not replace that account. It adds the other quantum computers that qBraid offers.

### IBM Quantum Platform in four steps

*Slide 5.*

IBM Quantum Platform is the website where you manage your IBM Quantum account.

1. **Sign in.** Go to quantum.cloud.ibm.com. Use or create an IBMid, the account you sign in to IBM's websites with.
2. **Check the header.** The right account is selected, and the region is us-east.
3. **Get an instance.** An instance is your share of access under a plan. If none is listed, create one on the Open Plan.
4. **Create an API key.** You do this on the dashboard. Copy the key now: it is shown once.

The Open Plan is free and gives up to 10 minutes of quantum time in every 28-day window (IBM Quantum documentation, Plans overview, read on Nov 13, 2026). A region is one location of IBM's data centers, and the header of the site shows the one you are using. Open Plan instances exist only in the us-east region.

Quantum time is the time a real processor spends running your jobs. Every instance has a CRN (Cloud Resource Name), a long identifier that says which instance a job belongs to. You copy it from the Instances page in the next step. The names of buttons on the site change from time to time, so look for these four ideas if a label reads differently.

### Save your IBM credentials once

*Slide 6.*

File on the slide: `save_account.py`

Your credentials are the API key and the instance CRN. You hand them to Qiskit one time. Qiskit stores them in a file in your home folder, and from then on every notebook on this computer can use them.

The steps on the slide:

1. Paste your API key
2. Paste your instance CRN, copied from the Instances page
3. Run the cell one time
4. Delete the key from the notebook

On the slide the key is typed into the cell and deleted afterwards. In this notebook the key never enters a cell, so the steps are:

1. Set `USE_HARDWARE = True` in the Setup section and run the first two setup cells again.
2. Run the cell below once. It opens two hidden boxes, one after the other. Paste your API key into the first and your instance CRN into the second. What you paste is not displayed and is not stored in the notebook.
3. From then on the key is read from your home folder. There is no key in the notebook, so the slide's step 4 has nothing to delete.

Your API key is confidential. Do not show it on a shared screen, and do not put it in a file that you share or upload, for example to GitHub. Naming your Open Plan instance makes sure that your jobs use it, which matters if you also have another instance.

What you should see. With `USE_HARDWARE = False` the cell asks for nothing and prints the one backend of the simulator, `[<SimulatedBackend('fake_fez')>]`. With your own account it prints the list of real IBM processors you can use.

If IBM later rejects the saved key, create a new key on the website, set `REPLACE_SAVED_KEY = True` in the cell, and run it again.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
from getpass import getpass
from qiskit_ibm_runtime import (
    QiskitRuntimeService)

REPLACE_SAVED_KEY = False     # True replaces a key that IBM rejects
if USE_HARDWARE and str(HARDWARE).strip().lower() == "ibm" and (
        REPLACE_SAVED_KEY or not QiskitRuntimeService.saved_accounts()):
    QiskitRuntimeService.save_account(
        token=getpass("IBM Quantum API key: "),
        instance=getpass("Instance CRN: "),
        set_as_default=True,
        overwrite=True)

# from now on, this is all you need
service = connect("ibm")
print(service.backends())

### Check your setup in one cell

*Slide 8.*

File on the slide: `check_setup.py`

The steps on the slide:

1. Run the install line if Qiskit is missing
2. Restart the kernel
3. Run the check
4. Hands down when you see a backend name

Steps 1 and 2 already happened in the Setup section: the setup cell installs the tested versions and asks for the restart. That is why the slide's install line is not in the cell below. Run the cell and read its last line.

- On the simulator the last line is `fake_fez 156`: the name of the simulated processor and its number of qubits.
- With `USE_HARDWARE = True` and your own account it is the name of a real IBM processor and its number of qubits. `least_busy` asks IBM for a working processor with the fewest jobs waiting.

The first two lines are the installed versions of Qiskit and of qiskit-ibm-runtime, the package that talks to IBM's processors. The line after them comes from `connect` and says whether the cell uses the simulator or your own account.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
import qiskit, qiskit_ibm_runtime
print(qiskit.__version__)
print(qiskit_ibm_runtime.__version__)

from qiskit_ibm_runtime import (
    QiskitRuntimeService)
service = connect("ibm")
backend = service.least_busy(
    operational=True, simulator=False)
print(backend.name, backend.num_qubits)

### The second option: qBraid in four steps

*Slide 7.*

qBraid Lab is a Jupyter workspace that runs in your browser. Jupyter is the program that shows and runs notebooks like this one. qBraid Lab is where the second option starts.

1. **Create an account.** account.qbraid.com. Use the email you registered with.
2. **Launch Lab.** lab.qbraid.com. Create a qBraid API key if it asks.
3. **Pick an environment.** Choose the Qiskit environment, or install Qiskit in the first cell.
4. **Open the notebook.** Tonight's notebook, from the event page.

IBM processors use your IBM Quantum account. qBraid adds other quantum computers, once your access arrives by email.

Notes for these steps:

- Access is tied to the email on your registration form, so sign in with that email.
- An environment is a set of installed Python packages. In this notebook the setup cell does the installing for step 3: it checks for the tested versions and installs what is missing.
- qBraid's documentation gives three rules for notebooks in Lab. Install packages with `%pip`, not with `!pip`. Restart the kernel after installing. Select the `qbraid-sdk` kernel when you send jobs. The setup cell of this notebook follows the first two: it installs with `%pip` and then asks you to restart.
- This notebook needs Python 3.11 or newer. The setup cell stops with one clear line if the kernel is older.

### Option 2: other quantum computers through qBraid

*Slide 9.*

File on the slide: `qbraid_run.py`

The steps on the slide:

1. List the devices and their status
2. Pick one that is online
3. Submit the same circuit
4. Wait, then read the counts

The cell below is the second option in code. It builds a small two-qubit test circuit (a Bell pair, which Part 1 explains), lists the quantum computers your account can reach, and sends the circuit to the one you chose. "The same circuit" on the slide means this test circuit, the one tonight's labs also run on an IBM processor.

How to use the cell:

1. In the Setup section set `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, then run the first two setup cells again. The first time, the setup cell installs the `qbraid` package and asks for a restart.
2. Run the cell below. Outside qBraid Lab it first asks for your qBraid API key in a hidden box. Then it prints one line per device, with the device's id and its status, and stops with an error saying that the device `PASTE_DEVICE_ID` was not found. That stop is expected on the first run, because no device is chosen yet.
3. Copy the id of a device whose status reads `DeviceStatus.ONLINE` into `QBRAID_DEVICE` in the Setup section. Run that cell, then run the cell below again.
4. The cell waits for the job and prints the counts.

Five lines in the cell are not on the slide: `if provider:`, which skips the rest of the cell unless the second option is switched on, and the four from `formats = ...` to `set_target_program_type`. The circuit travels to the device as text. On the tested versions the `qbraid` package can only load the newer of its two text formats, OpenQASM 3, so the cell asks for that format whenever the device accepts it.

Two things to know about the waiting. `job.wait_for_final_state()` also returns when a job has failed, and the last line then stops with `ValueError: Counts data is not available.` A job that is put on hold (status `HOLD`) never ends the wait. In that case interrupt the kernel and ask an organizer.

This cell has no cached result, because nothing later in the notebook uses its counts. If the job waits too long, interrupt the kernel and go on.

This cell was checked against the `qbraid` 0.13.0 package with the network replaced by a stand-in. It has not been run with a real qBraid account.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
qc.h(0); qc.cx(0, 1)
qc.measure_all()

provider = connect("qbraid")   # key saved by Lab
if provider:
    for dev in provider.get_devices():
        print(dev.id, dev.status())

    DEVICE = QBRAID_DEVICE        # one from the list
    device = provider.get_device(DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")
    job = device.run(qc, shots=1024)
    job.wait_for_final_state()
    print(job.result().data.get_counts())

### If setup goes wrong

*Slide 10.*

| What you see | What to do |
|---|---|
| No instance listed after signing in | Create one on the Open Plan, in the us-east region |
| The key is rejected, or the account is not found | Create a new API key and save it again with overwrite=True |
| No backends come back | Check the instance CRN and the region |
| ModuleNotFoundError: qiskit_ibm_runtime | Run the install line, then restart the kernel |
| A qBraid job stops with an access message | Check that you signed in with the email you registered with, then ask an organizer |
| Still stuck after two minutes | Ask in the chat, use the simulator cells, fix the account after the session |

How two of these rows look in this notebook. "Save it again with overwrite=True" means: set `REPLACE_SAVED_KEY = True` in the cell under "Save your IBM credentials once" and run it again. "Run the install line" means: run the setup cell again. It installs what is missing and then asks for the restart.

## Part 1: The seminar ideas

This part follows tonight's slides in order. Each heading is the title of a slide. Under it you find a short recap, the math from the slide, and often a small cell to run. The sections called "Derivation" show the steps behind a result, one at a time.

### Ten years ago, a quantum computer went online

*Slide 11.*

- 2016: IBM put a five-qubit processor online for anyone to program.
- 2026: you write a circuit tonight and run it on real hardware tonight.

This is the theme of this year's Fall Fest: a decade of quantum computers on the cloud. "Online" and "on the cloud" mean the same thing here. The machine sits in a laboratory, you send it a program over the internet, and the result comes back. "Real hardware" means such a machine, as opposed to a simulator.

### What is quantum computing?

*Slide 12.*

- A computer that stores information in quantum states, not in bits.
- It works with three ideas: superposition, entanglement, and measurement.
- It is not a faster laptop. It is a different kind of machine, good at particular problems.

A bit is the unit of an ordinary computer: something that is either 0 or 1. A quantum computer stores its information in the state of its qubits. A state is the full description of a qubit, everything you need in order to predict what a measurement will give. That description is richer than one of two values. The picture on the slide shows the difference: a bit is one of two values, and a qubit is an arrow that can point to any place on a sphere. You meet that sphere below, under "A qubit is a point on a sphere".

Keep the last bullet in mind all week. A quantum computer will not replace a laptop. It is a special-purpose machine for a short list of problem types.

### Three ideas carry the whole field

*Slide 13.*

- **Superposition.** A qubit can be a blend of 0 and 1 until it is measured.
- **Entanglement.** Several qubits can share one state that no single qubit holds on its own.
- **Measurement.** Reading a qubit forces a 0 or a 1, with odds set by its amplitudes.

Tonight's session is these three ideas in order: make a superposition, entangle two qubits, measure and count. The amplitudes in the third card are the numbers that describe the blend. They are defined below, under "A qubit is two complex numbers". Tuesday adds a fourth idea, interference, which is how algorithms steer the odds toward the right answer.

### Bits and qubits, side by side

*Slide 14.*

- **A bit.** Always 0 or 1. Reading it changes nothing. $n$ bits hold one of $2^n$ values.
- **A qubit.** A blend of 0 and 1 until measured. Reading it forces a 0 or a 1. $n$ qubits take $2^n$ amplitudes to describe.

You still read out only $n$ bits at the end. The skill is arranging the amplitudes first.

The last line of each card is the whole story. Take $n = 3$. Three bits hold one of the $2^3 = 8$ values from 000 to 111. Three qubits carry an amplitude for every one of those eight values at once. A measurement still returns only one of the eight, as three ordinary bits. A quantum algorithm is a way of arranging the amplitudes so that the value you read is likely to be a useful one.

### How a quantum program runs

*Slide 15.*

1. **Prepare.** Every qubit starts in 0.
2. **Apply gates.** Rotate and entangle the qubits.
3. **Measure.** Each qubit gives a 0 or a 1.
4. **Repeat.** Run it many times and count the results.

Steps 1 to 3 are one run of a circuit, and one run is called a shot. A single shot gives one bitstring: a string of 0s and 1s with one character per qubit, such as `01`. One bitstring tells you little, because the result of a measurement is random. The pattern shows up only in the counts, the number of shots that gave each bitstring. That is why every lab tonight asks for about a thousand shots.

### Why we want it: some problems outgrow every classical computer

*Slide 16.*

- Describing $n$ quantum particles takes $2^n$ numbers.
- Each added particle doubles the memory.
- A quantum computer holds that state in the hardware itself.

A classical computer is any ordinary computer, from a phone to a supercomputer. To simulate $n$ qubits it has to store all $2^n$ amplitudes. One amplitude takes 16 bytes of memory, so the total is

$$\text{memory} = 16 \times 2^{n} \ \text{bytes}$$

For 30 qubits that is $16 \times 2^{30}$ bytes, or 16 GiB. A gibibyte (GiB) is $2^{30}$ bytes, a little more than a billion bytes, so this is the memory of a well-equipped laptop. Every ten more qubits multiply the memory by $2^{10} = 1{,}024$: 16 TiB at 40 qubits and 16 PiB at 50. TiB and PiB are the next two units, each 1,024 times the one before. A quantum computer does not store these numbers anywhere. Its qubits are the state. This is the original argument for building a quantum computer, and Thursday's session returns to it.

The cell below draws the memory against the number of qubits. The vertical axis is logarithmic: each mark is a thousand times the mark below it. Look for the three marked points.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

qubits = np.arange(1, 61)                  # 1 to 60 qubits
bytes_needed = 16.0 * 2.0 ** qubits        # 16 bytes for each of the 2**n amplitudes

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(qubits, bytes_needed, color="tab:blue")
for marked, label in [(30, "16 GiB"), (40, "16 TiB"), (50, "16 PiB")]:
    ax.plot(marked, 16.0 * 2.0 ** marked, "o", color="black")
    ax.annotate(label, (marked, 16.0 * 2.0 ** marked), textcoords="offset points", xytext=(-48, 4))
ax.set_xlabel("number of qubits")
ax.set_ylabel("memory needed (bytes)")
ax.set_title("Memory needed to store the state of n qubits")
ax.grid(True, alpha=0.3)
plt.show()

The line is straight on this axis, which is what doubling with every qubit looks like. Going from 30 qubits to 50 multiplies the memory by about a million.

### What it could be used for

*Slide 17.*

- **Chemistry and materials.** Simulate molecules and materials directly.
- **Optimization.** Routing, scheduling, and graph problems: Tuesday.
- **Cryptography.** Breaks some of today's encryption and offers new key exchange: Wednesday.
- **Machine learning.** Quantum kernels and classifiers: Thursday.

Most of this is still research. Today's machines are noisy, which is Friday's topic.

These are the applications people are working toward, and each later session shows one of them at small scale. Some words from the cards. Optimization means finding the best choice among very many. Key exchange is how two people agree on a secret key for encryption. A classifier is a program that sorts data into groups, and a quantum kernel is a way to compare two data points with a quantum circuit (it has nothing to do with the Jupyter kernel from the Setup section). Noisy means that gates and measurements go slightly wrong now and then.

### Where the field is today

*Slide 18.*

- Processors with more than a hundred qubits are online.
- Errors, not qubit count, are what limit them.
- You can run on one tonight, for free.

What was checked for this notebook: on Nov 13, 2026 test jobs ran on two IBM processors through an Open Plan account, ibm_fez and ibm_marrakesh. IBM's documentation names a third processor for Open Plan accounts, ibm_kingston. Each of the three has 156 qubits. The simulated processor of this notebook is a copy of ibm_fez.

The cell below prints the size of the simulated processor and draws its qubit map. Each dot is a qubit. Each line joins a pair of qubits that can interact directly, which means that a two-qubit gate can act on that pair. The list of these pairs is called the coupling map.

In [ ]:
import matplotlib.pyplot as plt

print("qubits on the simulated processor:", SIMULATED_BACKEND.num_qubits)
pairs = SIMULATED_BACKEND.coupling_map.get_edges()     # every pair is listed twice, once in each direction
print("pairs of qubits that can interact:", len(pairs) // 2)

# Where to draw each qubit. This processor has 8 rows of 16 qubits, joined by short rows of 4 qubits.
position = {}
for q in range(SIMULATED_BACKEND.num_qubits):
    row, k = divmod(q, 20)
    if k < 16:
        position[q] = (k, -2 * row)
    else:
        position[q] = (4 * (k - 16) + (3 if row % 2 == 0 else 1), -2 * row - 1)

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for a, b in pairs:                                     # one line for each pair
    ax.plot([position[a][0], position[b][0]], [position[a][1], position[b][1]], color="0.6", lw=2, zorder=1)
for q, (x, y) in position.items():                     # one numbered dot for each qubit
    ax.scatter(x, y, s=130, color="#1f4e9c", zorder=2)
    ax.text(x, y, str(q), ha="center", va="center", color="white", fontsize=6, zorder=3)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("Qubit map of the simulated processor")
plt.show()

The processor has 156 qubits and 176 connected pairs. No qubit has more than three neighbors. A two-qubit gate between qubits that are not joined by a line has to be passed along through the qubits in between, and that takes extra gates. Friday's session comes back to this.

### A qubit is two complex numbers

*Slide 19.*

$$\lvert \psi \rangle = \alpha \lvert 0 \rangle + \beta \lvert 1 \rangle, \quad \lvert \alpha \rvert^{2} + \lvert \beta \rvert^{2} = 1$$

Measure it and you get 0 with probability $\lvert \alpha \rvert^2$, 1 with probability $\lvert \beta \rvert^2$.

How to read this. The symbol $\lvert \psi \rangle$ is called a ket. It is a name for a state, here the state "psi". $\lvert 0 \rangle$ and $\lvert 1 \rangle$ are the two states in which a measurement is certain to give 0 or 1. The numbers $\alpha$ and $\beta$ are the amplitudes. They say how much of each of the two is in the blend. A state in which both amplitudes are different from zero is a superposition.

The rule that turns amplitudes into probabilities is called the Born rule: the probability of a result is the squared size of its amplitude. The two probabilities have to add up to one, which is the condition on the right. A classical bit is the special case where one of the two amplitudes is zero.

The amplitudes are complex numbers. A complex number has the form $x + iy$, where $x$ and $y$ are ordinary numbers and $i^2 = -1$. Its size is $\lvert x + iy \rvert = \sqrt{x^2 + y^2}$. If complex numbers are new to you, read on anyway. In most of tonight's examples the amplitudes are ordinary numbers, some of them negative, and the size of an ordinary number is the number without its sign.

#### Derivation: Kets are column vectors

*Slide 20.*

The ket notation is shorthand for a column of numbers. Everything a gate does is a matrix, a table of numbers, acting on that column.

The two basis states are the columns with a single 1. They are called basis states because every other state is built from them.

$$\lvert 0 \rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad \lvert 1 \rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Any qubit state is $\alpha$ times the first column plus $\beta$ times the second, so its column holds the two amplitudes.

$$\lvert \psi \rangle = \alpha\lvert 0 \rangle + \beta\lvert 1 \rangle = \alpha\begin{pmatrix} 1 \\ 0 \end{pmatrix} + \beta\begin{pmatrix} 0 \\ 1 \end{pmatrix} = \begin{pmatrix} \alpha \\ \beta \end{pmatrix}$$

The bra $\langle \psi \rvert$ is the same state written as a row, with every amplitude replaced by its complex conjugate. The conjugate of $x + iy$ is $x - iy$, and it is written with a star. For an ordinary number the conjugate changes nothing.

$$\langle \psi \rvert = \begin{pmatrix} \alpha^{\ast} & \beta^{\ast} \end{pmatrix}$$

A row times a column is a single number: multiply the entries in pairs and add. A number times its own conjugate is its squared size, $\alpha^{\ast}\alpha = \lvert \alpha \rvert^2$. So the bra of a state times its own ket is the sum of the two probabilities, which is 1. This is the normalization condition of the slide before, written as a product.

$$\langle \psi \vert \psi \rangle = \alpha^{\ast}\alpha + \beta^{\ast}\beta = \lvert \alpha \rvert^2 + \lvert \beta \rvert^2 = 1$$

#### Derivation: Worked example: reading probabilities off a state

*Slide 21.*

Take this state.

$$\lvert \psi \rangle = \frac{\sqrt{3}}{2}\lvert 0 \rangle + \frac{1}{2}\lvert 1 \rangle$$

The probability of reading 0 is the squared size of the amplitude of $\lvert 0 \rangle$.

$$P(0) = \left\lvert \frac{\sqrt{3}}{2} \right\rvert^2 = \frac{3}{4}$$

The probability of reading 1 is the squared size of the amplitude of $\lvert 1 \rangle$.

$$P(1) = \left\lvert \frac{1}{2} \right\rvert^2 = \frac{1}{4}$$

They always sum to one.

$$\frac{3}{4} + \frac{1}{4} = 1$$

With complex amplitudes the rule is the same: square the size.

The cell below checks this example with Qiskit. A `Statevector` holds the list of amplitudes of a state. The cell prints the two probabilities, then measures 1,000 fresh copies of the state and counts the results.

In [ ]:
from qiskit.quantum_info import Statevector

psi = Statevector([3 ** 0.5 / 2, 1 / 2])       # the amplitudes of 0 and of 1
print("probabilities of reading 0 and 1:", psi.probabilities())

psi.seed(2026)                                 # fixes the random numbers, so the text below stays true
shots = psi.sample_counts(1000)                # measure 1,000 fresh copies of the state
print("in 1,000 shots, 0 came up", shots["0"], "times and 1 came up", shots["1"], "times")

The probabilities are 0.75 and 0.25, as in the derivation. In the 1,000 shots, 0 came up 731 times and 1 came up 269 times. That is close to 750 and 250, and not equal to them. The difference is called shot noise. The section "Measurement turns amplitudes into counts" below says how large it should be.

### Amplitudes are complex: each one has a size and a phase

*Slide 22.*

$$\alpha = \lvert \alpha \rvert\,e^{i\varphi}$$

- The size sets the probability.
- The phase is invisible on a single measurement.
- Phases decide whether amplitudes add or cancel when gates combine them.
- $\lvert + \rangle$ and $\lvert - \rangle$ give the same coin flip and are still different states.

Draw a complex number $x + iy$ as an arrow in a plane, $x$ to the right and $y$ upward. The length of the arrow is the size $\lvert \alpha \rvert$. The angle $\varphi$ of the arrow, measured from the direction that points right, is the phase. The factor $e^{i\varphi}$ is the arrow of length 1 at that angle: $e^{i\varphi} = \cos\varphi + i\sin\varphi$. A phase of 0 gives $+1$ and a phase of $\pi$, half a turn, gives $-1$. So a minus sign is a phase.

The two states in the last bullet are

$$\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}, \quad \lvert - \rangle = \frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$$

All four amplitudes have the size $\frac{1}{\sqrt{2}}$, so both states give 0 or 1 with probability one half: a fair coin flip. The two states differ only in the phase of the amplitude of $\lvert 1 \rangle$, which is 0 in $\lvert + \rangle$ and $\pi$ in $\lvert - \rangle$.

One clarification. On this slide $\varphi$ is the phase of a single amplitude. Only the phase of one amplitude relative to the other can ever be observed: multiplying both amplitudes by the same factor $e^{i\varphi}$ changes no measurement at all. From the next slide on, $\varphi$ therefore means the relative phase, the phase of the amplitude of $\lvert 1 \rangle$ when the amplitude of $\lvert 0 \rangle$ is an ordinary positive number.

The cell below draws three amplitudes of the same size, $\frac{1}{\sqrt{2}}$, with three different phases. The arrow with phase 0 is the amplitude of $\lvert 1 \rangle$ in $\lvert + \rangle$. The arrow with phase $\pi$ is the amplitude of $\lvert 1 \rangle$ in $\lvert - \rangle$. Compare the lengths of the arrows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

size = 1 / np.sqrt(2)
phases = {"phase 0": 0.0, "phase pi/3": np.pi / 3, "phase pi": np.pi}

fig, ax = plt.subplots(figsize=(6, 3.5))
circle = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(circle), np.sin(circle), color="0.8")                # every number of size 1
for (label, phase), color in zip(phases.items(), ["tab:blue", "tab:green", "tab:red"]):
    amplitude = size * np.exp(1j * phase)                           # size times e^(i phase)
    ax.annotate("", xy=(amplitude.real, amplitude.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.plot([], [], color=color, label=label)                       # entry for the legend
    print(label, "| amplitude", np.round(amplitude, 3), "| size", round(abs(amplitude), 3),
          "| probability", round(abs(amplitude) ** 2, 3))
ax.axhline(0, color="0.5", lw=0.5)
ax.axvline(0, color="0.5", lw=0.5)
ax.set_aspect("equal")
ax.set_xlabel("real part")
ax.set_ylabel("imaginary part")
ax.set_title("An amplitude as an arrow")
ax.legend(loc="center left", bbox_to_anchor=(1.05, 0.5))
plt.show()

The three arrows have the same length, 0.707, so each of the three amplitudes gives the probability 0.5. The length of an arrow is the size of the amplitude and its angle is the phase. In Python `1j` is the number $i$, which is why the middle amplitude prints as `(0.354+0.612j)`.

The next cell shows that a relative phase has an effect even though a single measurement cannot see it. It takes $\lvert + \rangle$ and $\lvert - \rangle$, prints their probabilities, then applies one H gate to each and prints the probabilities again. H is a gate that you meet a few sections below.

In [ ]:
import numpy as np
from qiskit.circuit.library import HGate
from qiskit.quantum_info import Statevector

plus = Statevector.from_label("+")       # (|0> + |1>) / sqrt(2)
minus = Statevector.from_label("-")      # (|0> - |1>) / sqrt(2)
print("measured directly:  |+> gives", plus.probabilities(), "  |-> gives", minus.probabilities())
print("after one H gate:   |+> gives", np.round(plus.evolve(HGate()).probabilities(), 3),
      "  |-> gives", np.round(minus.evolve(HGate()).probabilities(), 3))

Measured directly, the two states cannot be told apart: both give `[0.5 0.5]`. After one H gate, $\lvert + \rangle$ always gives 0 and $\lvert - \rangle$ always gives 1. Two states that differ only in a phase behave differently one gate later. That is what the third bullet means by amplitudes that add or cancel.

### A qubit is a point on a sphere

*Slide 23.*

$$\lvert \psi \rangle = \cos\frac{\theta}{2}\lvert 0 \rangle + e^{i\varphi}\sin\frac{\theta}{2}\lvert 1 \rangle$$

- $\theta$ sets the odds, $\varphi$ sets the phase.
- The poles are $\lvert 0 \rangle$ and $\lvert 1 \rangle$. The equator is an equal superposition.
- Gates are rotations of the sphere. Measurement snaps the point to a pole.

This picture is called the Bloch sphere. Here is why two angles are enough. Two complex amplitudes are four ordinary numbers. The normalization condition removes one of them. A phase that multiplies both amplitudes changes no measurement, which removes another. Two numbers remain, and they can be taken as two angles: $\theta$ is the angle down from the north pole, and $\varphi$ is the angle around the vertical axis. This $\varphi$ is the relative phase of the slide before.

The probabilities follow from the Born rule.

$$P(0) = \cos^2\frac{\theta}{2}, \quad P(1) = \sin^2\frac{\theta}{2}$$

At the north pole $\theta = 0$ and the state is $\lvert 0 \rangle$. At the south pole $\theta = \pi$ and the state is $\lvert 1 \rangle$. On the equator $\theta = \frac{\pi}{2}$, both probabilities are one half, and $\varphi$ says where on the equator the state sits: $\lvert + \rangle$ at $\varphi = 0$ and $\lvert - \rangle$ at $\varphi = \pi$.

The cell below draws the state with $\theta = \frac{\pi}{3}$ and $\varphi = \frac{\pi}{2}$ and prints its two probabilities.

In [ ]:
import numpy as np
from qiskit.visualization import plot_bloch_vector

theta, phi = np.pi / 3, np.pi / 2        # the two angles. Change them and run the cell again.

print("P(0) = cos^2(theta/2) =", round(np.cos(theta / 2) ** 2, 3))
print("P(1) = sin^2(theta/2) =", round(np.sin(theta / 2) ** 2, 3))

# The point on the sphere, as x, y and z coordinates.
point = [np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)]
title = f"theta = {theta / np.pi:.2f} pi, phi = {phi / np.pi:.2f} pi"
plot_bloch_vector(point, title=title, figsize=(3.5, 3.5))

$P(0) = 0.75$ and $P(1) = 0.25$. These are the odds of the worked example, which is this state with $\varphi = 0$. Now change `phi` and run the cell again: the arrow moves around the sphere at the same height and both probabilities stay the same. Change `theta` and the arrow moves toward a pole, which changes the odds.

### Measurement turns amplitudes into counts

*Slide 24.*

$$P(x) = \left\lvert \langle x \vert \psi \rangle \right\rvert^{2}$$

1,024 shots of a fair superposition give about $512 \pm 16$ zeros.

This is the Born rule again, written so that it works for any number of qubits. $x$ is one possible result, a bitstring. $\langle x \vert \psi \rangle$ is the amplitude of that result in the state $\lvert \psi \rangle$: the bra of $x$ times the ket of the state, as in "Kets are column vectors". For one qubit, $\langle 0 \vert \psi \rangle = \alpha$ and $\langle 1 \vert \psi \rangle = \beta$.

A probability is not a count. A fair superposition is one with $P(0) = P(1) = \frac{1}{2}$, like a fair coin, and 1,024 flips of a fair coin rarely give 512 heads on the dot. The counts scatter around 512, and the typical distance from 512 is 16. So 490 or 530 zeros is normal, and 300 zeros is a bug. The derivation below shows where the 16 comes from.

#### Derivation: Shot noise: how far off can the counts be?

*Slide 25.*

Every shot is an independent trial that gives 0 with the same probability $p$. Count 1 for a shot that gives 0 and count 0 otherwise. On average one shot then counts $p$. With probability $p$ the count is 1, at a distance $1-p$ from that average, and with probability $1-p$ the count is 0, at a distance $p$. The variance, which is the average squared distance from the average, is therefore $p(1-p)^2 + (1-p)p^2 = p(1-p)$. For $N$ independent shots the averages add up and the variances add up. The number of zeros in $N$ shots is said to follow the binomial distribution.

The mean count is the number of shots times the probability.

$$\mu = Np = 1024 \times \frac{1}{2} = 512$$

The standard deviation $\sigma$ is the square root of the variance. It is the typical distance of a count from the mean: about two thirds of all repetitions land within one standard deviation.

$$\sigma = \sqrt{Np(1-p)} = \sqrt{1024 \times \frac{1}{4}} = 16$$

The relative error is the standard deviation divided by the mean.

$$\frac{\sigma}{\mu} = \frac{\sqrt{Np(1-p)}}{Np} = \sqrt{\frac{1-p}{Np}} = \frac{1}{\sqrt{N}} \quad \text{at } p = \frac{1}{2}$$

For 1,024 shots that is $\frac{1}{\sqrt{1024}} = \frac{1}{32} \approx 3.1\%$. Four times the shots halves it.

$$N = 4096: \quad \frac{1}{\sqrt{4096}} = \frac{1}{64} \approx 1.6\%$$

Precision improves only as the square root of the number of shots. To halve the scatter you need four times the shots.

The cell below repeats the experiment "1,024 shots of a fair superposition" 200 times and draws a histogram of the 200 counts of zeros. It uses two things that are explained later: the H gate, which turns $\lvert 0 \rangle$ into the fair superposition $\lvert + \rangle$, and a sampler, the Qiskit tool that runs a circuit for a number of shots and returns the measured bits. Look at where the bars sit relative to the shaded band, which marks 512 plus or minus 16.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

coin = QuantumCircuit(1)
coin.h(0)                  # H makes the fair superposition
coin.measure_all()

# One job of 200 x 1,024 shots, cut into 200 experiments of 1,024 shots each.
seeded_sampler = StatevectorSampler(seed=np.random.default_rng(2026))      # the seed fixes the random numbers
bits = seeded_sampler.run([coin], shots=200 * 1024).result()[0].data.meas.array
zeros = 1024 - bits.reshape(200, 1024).sum(axis=1, dtype=int)             # zeros in each experiment

print("first eight experiments:", zeros[:8])
print("mean of the 200 counts:", round(zeros.mean(), 1), "| standard deviation:", round(zeros.std(ddof=1), 1))
print("counts within 512 plus or minus 16:", (np.abs(zeros - 512) <= 16).sum(), "of 200")
print("smallest and largest count:", zeros.min(), "and", zeros.max())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.axvspan(512 - 16, 512 + 16, color="tab:orange", alpha=0.3, label="512 plus or minus 16")
ax.hist(zeros, bins=np.arange(448, 584, 8), color="tab:blue", edgecolor="white")
ax.axvline(512, color="black", label="mean 512")
ax.set_xlabel("zeros in 1,024 shots")
ax.set_ylabel("number of experiments")
ax.set_title("200 experiments of 1,024 shots each")
ax.legend(loc="upper left")
plt.show()

The first experiment gave 474 zeros and the next two gave 531. Over all 200 experiments the mean count is 514.5 and the standard deviation is 15.7, close to the predicted 512 and 16. 142 of the 200 counts fall inside the band, which is 71 percent, and every count lies between 465 and 554. Nothing came anywhere near 300.

### Gates are rotations

*Slide 26.*

- X flips: $\lvert 0 \rangle \leftrightarrow \lvert 1 \rangle$.
- Z flips the sign of $\lvert 1 \rangle$.
- H sends $\lvert 0 \rangle$ to $\lvert + \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}}$ and $\lvert 1 \rangle$ to $\lvert - \rangle = \frac{\lvert 0 \rangle - \lvert 1 \rangle}{\sqrt{2}}$.

A gate changes the state of a qubit without measuring it. A state is a column of two numbers, so a gate on one qubit is a table of four numbers, a 2 by 2 matrix, that multiplies the column. The slide shows the three matrices.

$$X = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}, \quad Z = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \quad H = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}$$

On the Bloch sphere each of the three is a half-turn. X is a half-turn about the x axis. Z is a half-turn about the z axis, the vertical one. H, the Hadamard gate, is a half-turn about the axis halfway between x and z. That is why $H = \frac{X + Z}{\sqrt{2}}$, and why applying H twice brings every state back to where it started.

The cell below asks Qiskit for the three matrices and checks the relation between them.

In [ ]:
import numpy as np
from qiskit.circuit.library import HGate, XGate, ZGate
from qiskit.quantum_info import Operator

X_mat = Operator(XGate()).data       # each gate as a 2 by 2 array of numbers
Z_mat = Operator(ZGate()).data
H_mat = Operator(HGate()).data

print("X =")
print(np.round(X_mat.real, 3))       # .real: the entries of these three matrices have no imaginary part
print("Z =")
print(np.round(Z_mat.real, 3))
print("H =")
print(np.round(H_mat.real, 3))
print("H equals (X + Z) / sqrt(2):", np.allclose(H_mat, (X_mat + Z_mat) / np.sqrt(2)))

The printed matrices are the ones above. 0.707 is $\frac{1}{\sqrt{2}}$, rounded to three decimals. `np.allclose` compares two arrays and ignores rounding errors in the last decimals, so `True` means the two sides are equal.

#### Derivation: X, Z, and H on the basis states

*Slide 27.*

To multiply a matrix by a column, take one row of the matrix at a time, multiply its entries with the entries of the column in pairs, and add. The top row gives the top entry of the result and the bottom row gives the bottom entry.

X flips the bit. For example, the top row of X times the column of $\lvert 0 \rangle$ is $0 \times 1 + 1 \times 0 = 0$ and the bottom row gives $1 \times 1 + 0 \times 0 = 1$. The result is the column of $\lvert 1 \rangle$.

$$X\lvert 0 \rangle = \lvert 1 \rangle, \quad X\lvert 1 \rangle = \lvert 0 \rangle$$

Z flips the phase of 1. It leaves $\lvert 0 \rangle$ alone and puts a minus sign in front of $\lvert 1 \rangle$. That minus sign is a phase of $\pi$. It changes no probability: a qubit in $-\lvert 1 \rangle$ still gives 1 every time.

$$Z\lvert 0 \rangle = \lvert 0 \rangle, \quad Z\lvert 1 \rangle = -\lvert 1 \rangle$$

H makes and unmakes superpositions. The next derivation works out the first of these two products entry by entry. "Unmakes" refers to the fact, also shown there, that H undoes itself, so $H\lvert + \rangle = \lvert 0 \rangle$ and $H\lvert - \rangle = \lvert 1 \rangle$.

$$H\lvert 0 \rangle = \lvert + \rangle, \quad H\lvert 1 \rangle = \lvert - \rangle$$

A phase flip between two H gates is a bit flip.

$$HZH = X$$

To see why, read the product from right to left, because the gate next to the ket acts first. Follow $\lvert 0 \rangle$ through the three gates. The first H gives $\lvert + \rangle$. Z flips the sign of the $\lvert 1 \rangle$ part, which turns $\lvert + \rangle$ into $\lvert - \rangle$. The last H turns $\lvert - \rangle$ into $\lvert 1 \rangle$.

$$H\lvert 0 \rangle = \lvert + \rangle, \qquad Z\lvert + \rangle = \lvert - \rangle, \qquad H\lvert - \rangle = \lvert 1 \rangle$$

In the same way $\lvert 1 \rangle$ goes to $\lvert - \rangle$, then to $\lvert + \rangle$, then to $\lvert 0 \rangle$. Together the three gates swap $\lvert 0 \rangle$ and $\lvert 1 \rangle$, which is what X does. Every state is built from the two basis states, so two gates that agree on both basis states are the same gate.

Z changes no probability by itself, yet between two H gates it turns 0 into 1. The "Try it" cell in Part 2 runs exactly this.

#### Derivation: H on a qubit, as a matrix times a vector

*Slide 28.*

Basis states are columns.

$$\lvert 0 \rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}, \quad \lvert 1 \rangle = \begin{pmatrix} 0 \\ 1 \end{pmatrix}$$

Apply H to $\lvert 0 \rangle$. The top row gives $1 \times 1 + 1 \times 0 = 1$ and the bottom row gives $1 \times 1 + (-1) \times 0 = 1$. The factor $\frac{1}{\sqrt{2}}$ stays in front.

$$H\lvert 0 \rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix}$$

Name the result. A column with two equal entries is an equal blend of the two basis columns.

$$\frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}} = \lvert + \rangle$$

H undoes itself. To multiply two matrices, multiply the left one by each column of the right one. The two factors $\frac{1}{\sqrt{2}}$ multiply to $\frac{1}{2}$.

$$H^2 = \frac{1}{2}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix} = \frac{1}{2}\begin{pmatrix} 2 & 0 \\ 0 & 2 \end{pmatrix} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}$$

The result is the identity matrix $I$, the matrix that leaves every column as it is. Every gate is a matrix and every state is a column, and applying a gate is a matrix multiplication. The last line is why two H gates in a row do nothing.

The cell below repeats both derivations with numbers. It reuses the three matrices from the cell above. In Python, `@` multiplies matrices.

In [ ]:
zero = np.array([1, 0])      # the column of |0>
one = np.array([0, 1])       # the column of |1>

print("X on |0> gives", np.round((X_mat @ zero).real, 3))
print("Z on |1> gives", np.round((Z_mat @ one).real, 3))
print("H on |0> gives", np.round((H_mat @ zero).real, 3))
print("H on |1> gives", np.round((H_mat @ one).real, 3))
print("H Z H equals X:", np.allclose(H_mat @ Z_mat @ H_mat, X_mat))
print("H H equals the identity:", np.allclose(H_mat @ H_mat, np.eye(2)))

Each line agrees with the derivations. `[0. 1.]` is the column of $\lvert 1 \rangle$, `[ 0. -1.]` is $-\lvert 1 \rangle$, and `[0.707 0.707]` and `[ 0.707 -0.707]` are $\lvert + \rangle$ and $\lvert - \rangle$.

### Rotation gates turn the sphere by any angle

*Slide 29.*

$$R_y(\theta)\lvert 0 \rangle = \cos\frac{\theta}{2}\lvert 0 \rangle + \sin\frac{\theta}{2}\lvert 1 \rangle$$

- $R_x$, $R_y$, and $R_z$ rotate about the three axes.
- X, Z, and H are rotations too: each is a half-turn. X and Z are half-turns about the x and z axes, so they equal $R_x(\pi)$ and $R_z(\pi)$ up to a global phase. H is a half-turn about the axis halfway between x and z, so it is a rotation of the sphere, but it is not an $R_x$, $R_y$ or $R_z$.
- Every single-qubit gate is some rotation of the sphere.

$R_y(\theta)$ turns the sphere by the angle $\theta$ about the y axis. Starting from the north pole $\lvert 0 \rangle$, it moves the state down toward the south pole, and the formula above is the state it reaches. Compare it with the formula of the Bloch sphere: this is the point at the angle $\theta$ from the pole, with $\varphi = 0$. So the probability of reading 1 is

$$P(1) = \sin^2\frac{\theta}{2}$$

Setting $\theta = \frac{\pi}{2}$ gives an equal superposition. Setting $\theta = \pi$ turns $\lvert 0 \rangle$ into $\lvert 1 \rangle$. On $\lvert 0 \rangle$ that is the same effect as X, but $R_y(\pi)$ is not the X gate: it sends $\lvert 1 \rangle$ to $-\lvert 0 \rangle$, with a minus sign that X does not have. The rotation that equals X up to a global phase is $R_x(\pi)$. A global phase is a factor of size 1, such as $-i$, in front of the whole state. It changes no measurement.

Rotation angles are the knobs that Tuesday's and Thursday's algorithms tune.

The cell below checks the statements about half-turns. It reuses `X_mat` and `Z_mat` from above. In Python, `-1j` is the number $-i$.

In [ ]:
from qiskit.circuit.library import RXGate, RYGate, RZGate

print("R_x(pi) equals -i times X:", np.allclose(RXGate(np.pi).to_matrix(), -1j * X_mat))
print("R_z(pi) equals -i times Z:", np.allclose(RZGate(np.pi).to_matrix(), -1j * Z_mat))
print("R_y(pi) =")
print(np.round(RYGate(np.pi).to_matrix().real, 3))

The first two lines print `True`: $R_x(\pi)$ and $R_z(\pi)$ are X and Z up to the global phase $-i$. The matrix of $R_y(\pi)$ has a $-1$ where X has a $+1$, so it is a different gate.

The picture on the slide is a one-qubit circuit with an $R_y$ rotation by $\theta$, followed by a measurement. The next cell builds that circuit and draws it. Read a circuit drawing from left to right. The line labeled `q` is the qubit, the box is the gate, and the dial is the measurement. The double line `meas` at the bottom carries the measured bit. The dashed line is a barrier, a separator that `measure_all` adds to the drawing. It does nothing to the qubit.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

angle = Parameter("theta")           # a named placeholder for a number that is filled in when the circuit runs
rotation = QuantumCircuit(1)         # a circuit with one qubit, which starts in 0
rotation.ry(angle, 0)                # R_y on qubit 0
rotation.measure_all()               # measure every qubit
rotation.draw("mpl")

The next cell runs this circuit at 25 angles between 0 and $2\pi$, with 1,024 shots at each angle. It plots the fraction of shots that gave 1 on top of the curve $\sin^2\frac{\theta}{2}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qiskit.primitives import StatevectorSampler

angles = np.linspace(0, 2 * np.pi, 25)                                # 25 angles from 0 to 2 pi
seeded_sampler = StatevectorSampler(seed=np.random.default_rng(28))   # the seed fixes the random numbers
data = seeded_sampler.run([(rotation, angles)], shots=1024).result()[0].data.meas
p_one = data.bitcount().mean(axis=-1)                                 # fraction of shots that gave 1, at each angle
print("largest gap between a point and the curve:", round(np.abs(p_one - np.sin(angles / 2) ** 2).max(), 3))

smooth = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(smooth, np.sin(smooth / 2) ** 2, color="0.5", label="sin^2(theta/2)")
ax.plot(angles, p_one, "o", color="tab:blue", label="1,024 shots per angle")
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels(["0", "pi/2", "pi", "3 pi/2", "2 pi"])
ax.set_xlabel("rotation angle theta")
ax.set_ylabel("fraction of shots that gave 1")
ax.set_title("R_y(theta) on a qubit that starts in 0")
ax.legend()
plt.show()

The points follow the curve. At $\theta = \pi$ every one of the 1,024 shots gave 1, and at $\theta = \frac{\pi}{2}$ about half of them did. The largest gap between a point and the curve is 0.036. That gap is shot noise. One standard deviation of such a fraction is at most $\frac{16}{1024} \approx 0.016$, and the largest of 25 gaps is typically about twice that.

Slide 30 is a cell to try. You find it at the start of Part 2, under "Try it: one qubit, three gates".

### Two qubits: four amplitudes

*Slide 31.*

$$\lvert \psi \rangle = a\lvert 00 \rangle + b\lvert 01 \rangle + c\lvert 10 \rangle + d\lvert 11 \rangle$$

- Two qubits have four basis states; $n$ qubits have $2^n$.
- A product state can be written as one qubit's state times the other's.
- An entangled state cannot.

With two qubits a measurement has four possible results, 00, 01, 10 and 11, and the state has one amplitude for each. The four amplitudes obey the same rule as before. The probability of a result is the squared size of its amplitude, and the four probabilities add up to one.

$$\lvert a \rvert^2 + \lvert b \rvert^2 + \lvert c \rvert^2 + \lvert d \rvert^2 = 1$$

In a product state each qubit has a state of its own, and you describe the pair by describing each qubit. In an entangled state that is impossible: only the pair as a whole has a state. The derivation below says what "times" means, and the cell after it gives a quick test that tells the two kinds apart. The picture on this slide is a two-qubit circuit with one CX gate, the gate of the slide after next.

A note on order, because it matters from here on. On the slides the left character of a label such as $\lvert 10 \rangle$ belongs to the first qubit. Qiskit numbers the qubits from 0 and prints qubit 0 as the right character. The recaps below say which convention each line uses.

#### Derivation: The tensor product builds two-qubit states

*Slide 32.*

All lines of this derivation use the slide convention: the first qubit is the left character.

Put two kets side by side. The symbol $\otimes$ is called the tensor product. For basis states it joins the two labels: the first qubit in 0 and the second qubit in 1 is the two-qubit state 01.

$$\lvert 0 \rangle \otimes \lvert 1 \rangle = \lvert 01 \rangle$$

Written as four amplitudes, for 00, 01, 10, 11 in that order, the state $\lvert 01 \rangle$ has a 1 in the second place and zeros elsewhere.

$$\lvert 01 \rangle \rightarrow (0, \; 1, \; 0, \; 0)$$

A product state. The tensor product multiplies out like ordinary brackets. Put the first qubit in $\lvert + \rangle$ and the second qubit in $\lvert 0 \rangle$.

$$\lvert + \rangle \otimes \lvert 0 \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}} \otimes \lvert 0 \rangle = \frac{\lvert 00 \rangle + \lvert 10 \rangle}{\sqrt{2}}$$

A state that does not factor. Here $\lvert a \rangle$ and $\lvert b \rangle$ stand for any two one-qubit states (they are not the amplitudes $a$ and $b$ of the slide before). No choice of them gives the state on the left. This is the definition of entanglement.

$$\frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}} \neq \lvert a \rangle \otimes \lvert b \rangle$$

Why it does not factor. Multiply out the most general product of two one-qubit states, with amplitudes $p$ and $q$ for the first qubit and $r$ and $s$ for the second.

$$\left( p\lvert 0 \rangle + q\lvert 1 \rangle \right) \otimes \left( r\lvert 0 \rangle + s\lvert 1 \rangle \right) = pr\lvert 00 \rangle + ps\lvert 01 \rangle + qr\lvert 10 \rangle + qs\lvert 11 \rangle$$

Compare this with the four amplitudes of the slide before: $a = pr$, $b = ps$, $c = qr$ and $d = qs$. Then

$$ad - bc = (pr)(qs) - (ps)(qr) = 0$$

So every product state has $ad - bc = 0$. The state above has $a = d = \frac{1}{\sqrt{2}}$ and $b = c = 0$, which gives $ad - bc = \frac{1}{2}$. That is not zero, so the state is not a product.

The test also works in the other direction: if $ad - bc = 0$, the state is a product. For $a \neq 0$, multiply out $\left( \lvert 0 \rangle + \frac{c}{a}\lvert 1 \rangle \right) \otimes \left( a\lvert 0 \rangle + b\lvert 1 \rangle \right)$. The first three amplitudes come out as $a$, $b$ and $c$, and the last one as $\frac{bc}{a}$, which equals $d$ when $ad = bc$. If $a = 0$, then $b = 0$ or $c = 0$, and the state is $\lvert 1 \rangle \otimes \left( c\lvert 0 \rangle + d\lvert 1 \rangle \right)$ or $\left( b\lvert 0 \rangle + d\lvert 1 \rangle \right) \otimes \lvert 1 \rangle$. So a two-qubit state is a product state exactly when $ad - bc = 0$.

The cell below applies the test to the two states of this derivation.

In [ ]:
import numpy as np

def product_test(amplitudes):
    """Return the size of ad - bc for the four amplitudes of 00, 01, 10, 11."""
    a, b, c, d = amplitudes
    return abs(a * d - b * c)

plus_zero = np.array([1, 0, 1, 0]) / np.sqrt(2)      # |+> (x) |0> = (|00> + |10>) / sqrt(2)
bell_state = np.array([1, 0, 0, 1]) / np.sqrt(2)     # (|00> + |11>) / sqrt(2)

print("|+> (x) |0>:", np.round(plus_zero, 3), " size of ad - bc:", round(product_test(plus_zero), 3))
print("Bell pair:  ", np.round(bell_state, 3), " size of ad - bc:", round(product_test(bell_state), 3))

The first state gives 0: it is a product state. The second gives 0.5: it is entangled. The test gives the same value in Qiskit's order, because swapping the two qubits only swaps $b$ and $c$.

### CX: flip the target when the control is 1

*Slide 33.*

| Control, target | After CX |
|---|---|
| 0 0 | 0 0 |
| 0 1 | 0 1 |
| 1 0 | 1 1 |
| 1 1 | 1 0 |

- The first qubit is the control, the second is the target.
- On plain 0 and 1 inputs it is the XOR gate of ordinary computing.
- Give the control a superposition and the two qubits become entangled.

CX, also called CNOT (controlled NOT), is a gate on two qubits. One of them is the control and the other is the target. If the control is 1, the target is flipped. If the control is 0, nothing happens. On the inputs 0 and 1 the control itself does not change. XOR, short for "exclusive or", is the operation on two ordinary bits that gives 1 when exactly one of them is 1. After a CX the target holds the XOR of the two input bits.

Now the third bullet. Give the control an equal superposition of 0 and 1 and leave the target in 0. The input is then a blend of the first row of the table (0 0) and the third row (1 0), and CX acts on both parts at once. The output is a blend of 0 0 and 1 1. The two qubits are entangled: this is the Bell pair of the next slides.

The cell below applies CX to the four basis states and prints the results as Qiskit shows them. It does this twice, because the table and tonight's labs write the gate in a different order. In the middle column the control is qubit 1, the left character, so the column reads like the table. In the right column the control is qubit 0, the right character. That is how the labs write the gate: `cx(0, 1)`.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

left_control = QuantumCircuit(2)
left_control.cx(1, 0)       # control: qubit 1, the left character. Target: qubit 0, the right character.
right_control = QuantumCircuit(2)
right_control.cx(0, 1)      # control: qubit 0, the right character. Target: qubit 1, the left character.

print("input   cx(1, 0)   cx(0, 1)")
for label in ["00", "01", "10", "11"]:
    start = Statevector.from_label(label)
    after_left = start.evolve(left_control).probabilities_dict()      # one result with probability 1
    after_right = start.evolve(right_control).probabilities_dict()
    print("  " + label + "       ", *after_left, "       ", *after_right)

The middle column is the table from the slide, row for row. The right column is the same gate with the control on the right: `01` goes to `11`, and `11` goes to `01`. Both columns say one thing: the target flips when the control is 1. When a result looks reversed, check which character is qubit 0.

### Other two-qubit gates

*Slide 34.*

- CZ flips the sign of $\lvert 11 \rangle$ and leaves everything else alone.
- SWAP exchanges the states of two qubits.
- A real processor offers one native two-qubit gate and builds the others from it.

As matrices, with rows and columns in the order 00, 01, 10, 11:

$$CZ = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & -1 \end{pmatrix}, \quad \text{SWAP} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \end{pmatrix}$$

In CZ the two qubits play the same role: only $\lvert 11 \rangle$ gets the minus sign, so it does not matter which qubit you call the control. SWAP leaves 00 and 11 alone and exchanges 01 with 10.

A native gate is one that a processor can perform directly. Every other two-qubit gate in your circuit is rewritten into the native one before the circuit runs. On the simulated processor of this notebook, and on the IBM processor it copies, the native two-qubit gate is CZ. The program that does the rewriting is called the transpiler. You meet it in Lab 2 and again on Friday. Every two-qubit gate that runs costs some fidelity, a number between 0 and 1 that says how close the real result is to the ideal one.

The picture on the slide is a circuit with a CZ gate and a SWAP gate. The cell below prints the two matrices and draws that circuit.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import CZGate, SwapGate
from qiskit.quantum_info import Operator

print("CZ =")
print(np.round(Operator(CZGate()).data.real).astype(int))
print("SWAP =")
print(np.round(Operator(SwapGate()).data.real).astype(int))

symbols = QuantumCircuit(2)
symbols.cz(0, 1)        # drawn as two dots joined by a line
symbols.swap(0, 1)      # drawn as two crosses joined by a line
symbols.draw("mpl")

### Two gates make a Bell pair

*Slide 35.*

$$\text{CX}\left( H \otimes I \right)\lvert 00 \rangle = \frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}}$$

H puts the first qubit in an equal superposition. CX then flips the second qubit wherever the first is 1.
Each qubit alone looks like a coin flip, yet the two always agree.

Read the left side from right to left. $H \otimes I$ means H on the first qubit and nothing on the second, where $I$ is the identity, the gate that changes nothing. Then CX acts, with the first qubit as the control. The state on the right is called a Bell pair.

What the state says: the only possible results are 00 and 11, each with probability one half. Look at one qubit alone and you see 0 or 1 at random, like a fair coin. Look at both and they always match.

This slide writes the first qubit as the left character. The Bell pair reads the same in Qiskit's order, because 00 and 11 do not change when the two characters are swapped.

#### Derivation: Building the Bell pair, one gate at a time

*Slide 36.*

This derivation writes the first qubit as the left character.

Start with both qubits in 0.

$$\lvert 00 \rangle$$

H on the first qubit puts it in $\lvert + \rangle$. The pair is still a product state, the one from the tensor product derivation.

$$(H \otimes I)\lvert 00 \rangle = \frac{\lvert 0 \rangle + \lvert 1 \rangle}{\sqrt{2}} \otimes \lvert 0 \rangle = \frac{\lvert 00 \rangle + \lvert 10 \rangle}{\sqrt{2}}$$

CX flips the second qubit where the first is 1. It acts on each part of the blend separately. In the part $\lvert 00 \rangle$ the control is 0, so that part stays. In the part $\lvert 10 \rangle$ the control is 1, so the target flips and the part becomes $\lvert 11 \rangle$.

$$CX\,\frac{\lvert 00 \rangle + \lvert 10 \rangle}{\sqrt{2}} = \frac{CX\lvert 00 \rangle + CX\lvert 10 \rangle}{\sqrt{2}} = \frac{\lvert 00 \rangle + \lvert 11 \rangle}{\sqrt{2}}$$

Measure. The probability of each result is the squared size of its amplitude.

$$P(00) = P(11) = \left\lvert \frac{1}{\sqrt{2}} \right\rvert^2 = \frac{1}{2}, \quad P(01) = P(10) = 0$$

The cell below builds the same circuit the way the labs do, with `h(0)` and `cx(0, 1)`, and prints the amplitudes that are not zero after each gate. The labels are the ones Qiskit prints, with qubit 0 as the right character.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def nonzero_amplitudes(circuit):
    """The amplitudes of a two-qubit circuit that are not zero, with the labels Qiskit prints."""
    state = Statevector(circuit)
    shown = {}
    for label, value in zip(["00", "01", "10", "11"], state.data):
        if abs(value) > 1e-9:
            shown[label] = round(float(value.real), 3)
    return shown

bell = QuantumCircuit(2)
print("start:             ", nonzero_amplitudes(bell))
bell.h(0)
print("after H on qubit 0:", nonzero_amplitudes(bell))
bell.cx(0, 1)
print("after CX:          ", nonzero_amplitudes(bell))
bell.draw("mpl")

After H the two labels are `00` and `01`: qubit 0, the right character, is in superposition. The derivation writes the same step as 00 and 10, because it puts the first qubit on the left. It is the same state with the characters in the other order. After CX the labels are `00` and `11` in both conventions.

In the drawing, H is the box on qubit 0. CX is the dot, which marks the control, joined to the circled plus, which marks the target.

The next cell adds the measurements, runs 1,024 shots, prints the first ten of them and draws the counts as a histogram.

In [ ]:
import numpy as np
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

measured_bell = bell.measure_all(inplace=False)                       # a copy of the circuit, with measurements
seeded_sampler = StatevectorSampler(seed=np.random.default_rng(35))   # the seed fixes the random numbers
bell_result = seeded_sampler.run([measured_bell], shots=1024).result()[0]
print("the first ten shots:", bell_result.data.meas.get_bitstrings()[:10])
bell_counts = bell_result.data.meas.get_counts()
print("counts:", bell_counts)
plot_histogram(bell_counts, title="1,024 shots of a Bell pair", figsize=(5, 3.2))

In each of the ten shots the two characters are equal. Over the 1,024 shots, `00` came up 534 times and `11` came up 490 times, and `01` and `10` never appeared. Which of the two results a shot gives is random. That both qubits give the same result is certain.

### A GHZ state is the same trick on $n$ qubits

*Slide 37.*

$$\lvert \text{GHZ}_n \rangle = \frac{\lvert 00 \ldots 0 \rangle + \lvert 11 \ldots 1 \rangle}{\sqrt{2}}$$

- One H, then $n - 1$ CX gates.
- Only two outcomes should ever appear.
- Every extra CX costs fidelity.

A GHZ state is a Bell pair grown to $n$ qubits: all zeros and all ones, with equal amplitudes. The letters are the initials of Greenberger, Horne and Zeilinger, the physicists it is named after. To build it, put the first qubit in superposition with one H. Then add a chain of CX gates, each with the newest qubit of the chain as control and one more qubit as target. An ideal machine gives only two results, $n$ zeros or $n$ ones. On a real machine every CX goes wrong now and then, so other results appear, and more of them as $n$ grows. That is what tonight's challenge is about.

The picture on the slide is a five-qubit GHZ circuit. The cell below builds one and draws it.

In [ ]:
from qiskit import QuantumCircuit

chain = QuantumCircuit(5)
chain.h(0)                     # one H
for i in range(4):             # then four CX gates, each one adding a qubit to the chain
    chain.cx(i, i + 1)
chain.draw("mpl")

#### Derivation: Why bigger is harder: the error exponent

*Slide 38.*

Call $\varepsilon$ the probability that one two-qubit gate fails. Then one CX succeeds with probability

$$1 - \varepsilon$$

The GHZ circuit on $n$ qubits has $n - 1$ CX gates. If the failures are independent of each other, the probability that all of them succeed is the product of the single probabilities. $F$ stands for fidelity.

$$F \approx (1 - \varepsilon)^{n-1}$$

Illustrative numbers. The value of $\varepsilon$ below, half a percent, is an example. The real two-qubit error is a property of the backend and differs from one pair of qubits to the next.

$$\varepsilon = 0.005, \; n = 20: \quad 0.995^{19} \approx 0.909$$

A rule of thumb for small errors. For a small $\varepsilon$, $\ln(1 - \varepsilon) \approx -\varepsilon$, so

$$(1 - \varepsilon)^{n-1} = e^{(n-1)\ln(1 - \varepsilon)} \approx e^{-(n-1)\varepsilon}$$

With the numbers above, $e^{-19 \times 0.005} = e^{-0.095} \approx 0.909$. The estimate falls exponentially with the number of qubits. This exponent is what the challenge is a fight against, and it is why the choice of physical qubits matters. A physical qubit is one particular qubit on the chip, one dot on the qubit map. The transpiler places each qubit of your circuit on one of them, and some pairs of physical qubits have a smaller $\varepsilon$ than others.

The estimate leaves one thing out. It counts only the two-qubit gates. Reading a qubit out also goes wrong now and then, which is called readout error. On the simulated processor of this notebook readout errors cost more than the gates do: a 20-qubit GHZ state there gives all zeros or all ones in about 0.77 of the shots, against the 0.909 of the estimate.

The cell below draws the estimate and the rule of thumb for 2 to 100 qubits.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

eps = 0.005                                    # illustrative chance that one CX fails
sizes = np.arange(2, 101)                      # n from 2 to 100 qubits
estimate = (1 - eps) ** (sizes - 1)
rule_of_thumb = np.exp(-(sizes - 1) * eps)
print("n = 20: estimate", round(0.995 ** 19, 3), "| rule of thumb", round(np.exp(-19 * 0.005), 3))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(sizes, estimate, color="tab:blue", label="(1 - eps)^(n - 1)")
ax.plot(sizes, rule_of_thumb, "--", color="tab:orange", label="exp(-(n - 1) eps)")
ax.plot(20, 0.995 ** 19, "o", color="black")
ax.annotate("n = 20: 0.909", (20, 0.995 ** 19), textcoords="offset points", xytext=(8, 6))
ax.set_ylim(0, 1.05)
ax.set_xlabel("number of qubits n")
ax.set_ylabel("estimated chance that every CX works")
ax.set_title("GHZ estimate for eps = 0.005")
ax.legend()
plt.show()

The two curves lie on top of each other, which is the rule of thumb at work. At 100 qubits the estimate is still above one half. "Your turn 3" in Part 2 asks where it crosses one half.

### Every Qiskit program has four steps

*Slide 39.*

1. **Map.** Turn the problem into a circuit.
2. **Optimize.** Rewrite it for one specific backend.
3. **Execute.** Run it with a primitive: Sampler for counts, Estimator for expectation values.
4. **Post-process.** Turn the result into an answer.

Every lab this week uses these four headings. In tonight's Lab 2 they are the comments `# 1 Map` to `# 4 Post-process` in the code.

- **Map.** Write the circuit that expresses your problem. Tonight the problem is to make a Bell pair or a GHZ state, so the circuit is a few gates.
- **Optimize.** A backend is the machine, real or simulated, that runs a circuit. Each backend has its own native gates and its own qubit map. Rewriting a circuit so that it uses only those gates and those connections is called transpiling. In Qiskit the object that does it is called a pass manager: `pm.run(circuit)` returns the rewritten circuit.
- **Execute.** A primitive is Qiskit's standard interface for running circuits: you hand it circuits and a number of shots, and it hands back results. There are two. The Sampler returns the measured bitstrings, from which you get counts. The Estimator returns expectation values. An expectation value is the average of a measured quantity over many shots. Tonight uses only the Sampler.
- **Post-process.** Turn the raw result into the answer you wanted, for example from counts to the fraction of shots in which two qubits agree.

## Part 2: Hands-on

The code in the "Try it" and "Lab" cells is the code on the slides. Run each cell, read what it prints, then change something and run it again. After each lab comes a "Your turn" exercise with a check cell.

### Try it: one qubit, three gates

*Slide 30.*

File on the slide: `hzh.py`

1. Predict the result before you run it
2. H takes the qubit to the equator
3. Z flips its phase
4. H brings it back, to the other pole

The equator and the poles in these steps are those of the Bloch sphere. `Statevector(qc)` computes the state at the end of the circuit exactly. Nothing is measured and there are no shots. Before you run the cell, predict the two amplitudes it will print.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

qc = QuantumCircuit(1)
qc.h(0)      # to the equator
qc.z(0)      # flip the phase
qc.h(0)      # back to a pole

print(Statevector(qc))
# amplitudes (0, 1): the qubit is now 1
# H, then Z, then H acts exactly like X

The output lists the two amplitudes as complex numbers. The first is 0 and the second is 1, so the qubit is in $\lvert 1 \rangle$, as the last two comments in the cell say. Depending on your computer the amplitudes are printed as `0.+0.j` and `1.+0.j`, or as something like `-2.23711432e-17+0.j` and `1.00000000e+00+0.j`. A number that ends in `e-17` is about $10^{-17}$. It is a rounding error of ordinary computer arithmetic and stands for zero. The `+0.j` says that the imaginary part is zero.

Z alone changes no probability, yet between two H gates it turns 0 into 1. The relative phase has a visible effect.

### Lab 1: a Bell pair on the simulator

*Slide 41.*

File on the slide: `bell_sim.py`

1. Open tonight's notebook
2. Run the setup cell
3. Build the circuit
4. Sample 1,024 shots
5. Expect only 00 and 11

Steps 1 and 2 are done once you have run the Setup section. The cell below is steps 3 to 5. Line by line:

- `QuantumCircuit(2)` makes a circuit with two qubits, both starting in 0.
- `qc.h(0)` and `qc.cx(0, 1)` are the two gates of the Bell pair: H on qubit 0, then CX with qubit 0 as the control and qubit 1 as the target.
- `qc.measure_all()` adds a measurement of every qubit. The measured bits are stored under the name `meas`, which is why the result is read from `result.data.meas`.
- `StatevectorSampler()` is a Sampler that runs on the computer that runs this notebook. It computes the exact state and draws the shots from it, so it behaves like an ideal quantum computer, one without noise.
- `sampler.run([qc], shots=1024)` starts a job with one circuit and 1,024 shots. `job.result()` waits for the job and returns one result for each circuit, so `[0]` picks the result of ours.
- `get_counts()` turns the measured bits into a dictionary that maps each bitstring to its number of shots.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

qc = QuantumCircuit(2)
qc.h(0)               # superposition
qc.cx(0, 1)           # entangle
qc.measure_all()

sampler = StatevectorSampler()
job = sampler.run([qc], shots=1024)
result = job.result()[0]
counts = result.data.meas.get_counts()
print(counts)         # only '00' and '11'

Only `00` and `11` appear. Each count is near 512 and the two add up to 1,024. Your numbers differ from everyone else's, and they change every time you run the cell. That is shot noise: counts between about 480 and 544, two standard deviations around 512, are ordinary. The order of the two entries can change as well.

#### Your turn 1

Build a Bell pair whose two qubits always disagree: only `01` and `10` may appear, each with probability one half. Add your gates to the circuit `anti`, in place of the line `# YOUR CODE HERE`. Gates from Part 1 are enough. You do not need to add measurements, because the check cell computes the probabilities from the circuit.

In [ ]:
from qiskit import QuantumCircuit

anti = QuantumCircuit(2)
# YOUR CODE HERE
anti.draw("mpl")

In [ ]:
from qiskit.quantum_info import Statevector

assert anti.size() > 0, "The circuit is still empty. Add gates to anti in the cell above."
p = Statevector(anti.remove_final_measurements(inplace=False)).probabilities_dict()
assert max(p.get("00", 0), p.get("11", 0)) < 1e-9, "00 and 11 must never appear. Start from the Lab 1 pair, then change one qubit with a gate from the seminar."
assert abs(p.get("01", 0) - 0.5) < 1e-9 and abs(p.get("10", 0) - 0.5) < 1e-9, "01 and 10 should each have probability one half."
print("Check passed: only 01 and 10 appear, each with probability one half.")

### Lab 2: the same circuit on real hardware

*Slide 42.*

File on the slide: `bell_hardware.py`

1. Pick the least busy backend
2. Transpile for it
3. Submit the job
4. While it queues, predict the counts
5. Compare with the simulator

This is the Bell pair again, taken through all four steps of a Qiskit program. The comments in the code mark the steps.

- `# 1 Map`: the same circuit as in Lab 1.
- `# 2 Optimize`: `connect("ibm")` returns the service, as described in the Setup section. `least_busy` picks a backend. `generate_preset_pass_manager` builds a pass manager for that backend, and `optimization_level=1` asks it for light optimization. `pm.run(qc)` transpiles the circuit. The result is named `isa` because a circuit written in a backend's own gates is called an ISA circuit, for "instruction set architecture".
- `# 3 Execute`: `SamplerV2(mode=backend)` is the Sampler for IBM backends. It comes from the package qiskit-ibm-runtime. `sampler.run([isa], shots=1024)` submits the job.
- `# 4 Post-process`: read the counts, as in Lab 1.

With `USE_HARDWARE = False` everything happens on the computer that runs this notebook, and nothing is sent. The backend is then the simulated processor, which imitates the errors of the IBM processor ibm_fez, and the cell takes a few seconds. With `USE_HARDWARE = True` the job goes to a real IBM processor and waits in its queue, the line of jobs that arrived before yours. While it waits, do step 4 of the slide and predict the counts. When this cell was run on the IBM processor ibm_marrakesh on Nov 13, 2026, the job was done about 10 seconds after it started and used about 2 seconds of quantum time. That is one measurement. Queues vary.

The cell prints only the line from `connect`. The counts are shown a little further down, next to those of Lab 1.

In [ ]:
# Glue for the comparison after Lab 2: keep the Lab 1 result under its own name.
# The lab cell below reuses the names qc, sampler, job, result and counts.
assert "counts" in globals(), "There is no Lab 1 result yet. Run the Lab 1 cell first, then this cell."
assert set(counts) <= {"00", "11"}, "counts no longer holds the Lab 1 result. Run Lab 1 again, then this cell."
counts_sim = counts

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import (
    generate_preset_pass_manager)
from qiskit_ibm_runtime import (
    QiskitRuntimeService, SamplerV2)
qc = QuantumCircuit(2)            # 1 Map
qc.h(0); qc.cx(0, 1)
qc.measure_all()
service = connect("ibm")          # 2 Optimize
backend = service.least_busy(
    operational=True, simulator=False)
pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend)
isa = pm.run(qc)
sampler = SamplerV2(mode=backend) # 3 Execute
job = sampler.run([isa], shots=1024)
result = job.result()[0]          # 4 Post-process
counts = result.data.meas.get_counts()

**If your job is stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Then change `USE_CACHED` to `True` in the next cell and run it. It loads a result that was recorded on a real IBM processor, so the rest of the notebook works. Interrupting stops the waiting, not the job: the job stays in IBM's queue. If your job finished, leave `USE_CACHED = False`.

In [ ]:
# Cached result for the cell above, in case your job is still waiting in the queue.
# Recorded Nov 13, 2026 on the IBM processor ibm_marrakesh (1,024 shots).
CACHED_BELL = {"00": 507, "11": 430, "01": 82, "10": 5}
USE_CACHED = False      # change to True only if your job is stuck
if USE_CACHED:
    counts = dict(CACHED_BELL)
    print("Loaded the cached result:", counts)
elif "counts" not in globals() or counts is globals().get("counts_sim"):
    print("The cell above did not finish, so there is no processor result. Run it again, or set USE_CACHED = True and run this cell again.")
elif counts == CACHED_BELL:
    print("Cache not loaded this time. counts still holds the cached result you loaded earlier.")
else:
    print("Cache not loaded. The result from the cell above is kept.")

Step 5 of the slide: compare with the simulator.

In [ ]:
counts_hw = counts      # the Lab 2 result under its own name, because Lab 3 reuses the name counts
if counts_hw is counts_sim:
    print("Lab 2 has no processor result yet, so both lines below show the Lab 1 result. Run Lab 2 again, or load the cached result.")
print("Lab 1, ideal simulator:", counts_sim)
print("Lab 2, processor:      ", counts_hw)

Lab 1 gave only `00` and `11`. Lab 2 gives mostly `00` and `11`, plus a few `01` and `10`. Where do those come from? From noise: on a real processor the gates and the readout go wrong now and then, and the simulated processor imitates that. Noise is the whole of Friday's session.

On the simulated processor expect a few percent of the shots in `01` and `10` together. In the cached result from the real processor ibm_marrakesh it is 87 of the 1,024 shots, about 8.5 percent, and the errors are uneven: 82 shots of `01` against 5 of `10`, and fewer `11` than `00` (430 against 507). That pattern fits one of the two qubits reading 0 where a 1 was expected far more often than the other one did.

The next cell shows what step 2 did. It draws the transpiled circuit. `idle_wires=False` hides the qubits of the processor that the circuit does not use.

In [ ]:
isa.draw("mpl", idle_wires=False, scale=0.8)

This circuit does the same job as your H and CX, written with the gates the backend has. On the simulated processor, and on the Open Plan processors checked for this notebook, those are `rz`, `sx` and `x` on single qubits and `cz` on a pair. This circuit needs only `rz`, `sx` and `cz`. `rz` is the rotation $R_z$ from Part 1. `sx` is the square root of X, drawn as $\sqrt{X}$: two of them in a row make an X. The labels on the left say which physical qubit of the processor each of your two qubits was placed on. "Global phase" at the top is a phase factor on the whole state. It changes no measurement.

#### Your turn 2

Complete the function `agreement`. It takes a dictionary of counts for two qubits and returns the fraction of shots in which both qubits gave the same bit: the shots that read `00` or `11`, divided by all shots. The fraction is a number between 0 and 1. Careful: an outcome that never occurred is missing from the dictionary.

In [ ]:
def agreement(counts):
    """Return the fraction of shots in which the two qubits gave the same bit."""
    frac = None
    # YOUR CODE HERE
    return frac

In [ ]:
try:
    first = agreement({"00": 1, "11": 1, "01": 1, "10": 1})
except TypeError as error:
    first = error
assert not isinstance(first, TypeError), f"agreement() stopped with a TypeError ({first}). If you wrote sum(counts): that adds up the keys of the dictionary, which are strings. Add up the numbers of shots instead."
assert first is not None, "agreement() still returns None. Replace the line YOUR CODE HERE with code that sets frac."
examples = [
    ({"00": 30, "11": 50, "01": 15, "10": 5}, 0.8),
    ({"00": 512, "11": 512}, 1.0),
    ({"01": 3, "10": 1}, 0.0),
    ({"00": 1, "01": 3}, 0.25),
]
for example, want in examples:
    try:
        got = agreement(example)
    except (KeyError, TypeError):
        got = None
    assert got is not None, "An outcome that never occurred is missing from the dictionary. counts.get(key, 0) returns 0 for it."
    assert abs(got - want) < 1e-9, f"For {example} the qubits agree in {want:.0%} of the shots, so the fraction is {want}. You returned {got}."
print("Check passed.")
print("Lab 1, ideal simulator:", agreement(counts_sim))
print("Lab 2, processor:      ", round(agreement(counts_hw), 3))

### Lab 3: grow it into a GHZ state

*Slide 43.*

File on the slide: `ghz.py`

1. Add one qubit and one CX at a time
2. Rerun on the simulator
3. Pick $n$ and send it to hardware

`ghz(n)` builds the GHZ circuit for any number of qubits `n`: one H, then a loop that adds one CX for each further qubit. It is the chain drawn in Part 1. The cell then runs `ghz(5)` on the ideal simulator.

The code on the slide reuses the name `sampler`. After Lab 2 that name points at the Sampler of the processor, which accepts only transpiled circuits. The small cell before the lab code points `sampler` at the ideal simulator again. Without it the lab cell stops with `AerError: 'unknown instruction: h'`.

In [ ]:
# Glue for the next cell: Lab 2 pointed the name sampler at the processor.
# This lab starts on the ideal simulator again, as in Lab 1.
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

In [ ]:
def ghz(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)   # one more qubit joins
    qc.measure_all()
    return qc

qc = ghz(5)
job = sampler.run([qc], shots=1024)
result = job.result()[0]
counts = result.data.meas.get_counts()
print(counts)   # '00000' and '11111'

# next: choose n, transpile, run on hardware

Only `00000` and `11111` appear, about 512 each. For steps 1 and 2 of the slide, change the 5 in `qc = ghz(5)` to 6, then 7, and run the cell again each time. The two results get longer and there are still only two of them. Stay at 20 qubits or fewer in this cell. The ideal simulator stores all $2^n$ amplitudes, so every extra qubit doubles its work: on the laptop used to test this notebook, 20 qubits took about 8 seconds.

Step 3 of the slide is to pick $n$ and send the circuit to hardware. The slide's code ends with the comment `# next: choose n, transpile, run on hardware`. The cell below does that. It is not on the slide. It picks the backend and builds the pass manager `pm` the way Lab 2 does.

- `isa = pm.run(ghz(n))` transpiles the circuit for the backend.
- The lines about `mode` matter only on the simulated processor. Its default method stores every amplitude and becomes slow near 16 qubits (about 8 seconds at 16 and about 29 seconds at 18 on the test laptop). The matrix-product-state method stores the state as a chain of small tables of numbers, one per qubit. The tables stay small when little entanglement crosses each point of the chain. In a GHZ state that amount stays the same however long the chain is, so the method handles 60 qubits in seconds. Thursday's session explains why.
- The last two lines count the shots that came out as all zeros or all ones and print them as a fraction of the 1,024 shots.

What to expect: about 0.94 on the simulated processor at `n = 5`, and less as `n` grows. Change `n` and run the cell again to see it fall.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2

service = connect("ibm")                   # your account, or the simulated processor
backend = service.least_busy(operational=True, simulator=False)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
n = 5                                      # choose n
isa = pm.run(ghz(n))                       # transpile for that backend
mode = backend
if type(service).__name__ == "SimulatedService":  # simulated processor: a method that copes with many qubits
    mode = AerSimulator.from_backend(backend, method="matrix_product_state")
ghz_sampler = SamplerV2(mode=mode)
job = ghz_sampler.run([isa], shots=1024)
ghz_counts = job.result()[0].data.meas.get_counts()
both = ghz_counts.get("0" * n, 0) + ghz_counts.get("1" * n, 0)
print("fraction of shots that are all zeros or all ones:", both / 1024)

**If your job is stuck in the queue.** Interrupt the kernel, change `USE_CACHED` to `True` in the next cell and run it, as in Lab 2. The cached result is a five-qubit GHZ state measured on a real IBM processor. In it, 0.870 of the shots are all zeros or all ones.

In [ ]:
# Cached result for the cell above, in case your job is still waiting in the queue.
# Recorded Nov 13, 2026 on the IBM processor ibm_fez (GHZ state on 5 qubits, 1,024 shots).
CACHED_GHZ = {
    "00000": 450, "11111": 441, "11110": 25, "00111": 18, "01000": 17, "00001": 16, "01111": 14,
    "10000": 9, "11101": 7, "10111": 5, "00011": 4, "11000": 4, "00010": 2, "00100": 2,
    "01001": 2, "11011": 2, "11100": 2, "00101": 1, "00110": 1, "10001": 1, "10101": 1,
}
USE_CACHED = False      # change to True only if your job is stuck
if USE_CACHED:
    n = 5
    ghz_counts = dict(CACHED_GHZ)
    both = ghz_counts.get("0" * n, 0) + ghz_counts.get("1" * n, 0)
    print("Loaded the cached result. Fraction of shots that are all zeros or all ones:", both / 1024)
elif "ghz_counts" not in globals():
    print("The cell above did not finish, so there is no result yet. Run it again, or set USE_CACHED = True and run this cell again.")
elif ghz_counts == CACHED_GHZ:
    print("Cache not loaded this time. ghz_counts still holds the cached result you loaded earlier.")
else:
    print("Cache not loaded. The result from the cell above is kept.")

The next cell lists the six most frequent results of your GHZ run.

In [ ]:
for outcome in sorted(ghz_counts, key=ghz_counts.get, reverse=True)[:6]:
    print(outcome, ghz_counts[outcome])

With `n = 5` the first two lines are the two results a GHZ state should give. Every other line is an error.

The second option sends the same GHZ circuit to another quantum computer. `isa` was rewritten for one IBM processor, so it does not fit a different machine. The cell below therefore sends `ghz(n)`, the circuit before transpiling. It works like the cell in the Accounts section: it runs only with `USE_HARDWARE = True` and `HARDWARE = "qbraid"`, it prints the device list, and it needs the id of an online device in `QBRAID_DEVICE`. The two warnings about waiting apply here too. Like that cell, it has not been run with a real qBraid account, and it has no cached result: nothing later in the notebook uses its counts. If the job waits too long, interrupt the kernel and go on.

**Hardware cell, second option (qBraid).** With `USE_HARDWARE = True` and `HARDWARE = "qbraid"` this cell uses your qBraid account. With any other setting it is skipped and uses no account.

In [ ]:
provider = connect("qbraid")
if provider:
    for dev in provider.get_devices():
        print(dev.id, dev.status())

    device = provider.get_device(QBRAID_DEVICE)
    formats = device.profile.program_spec   # the text formats this device accepts
    formats = formats if isinstance(formats, list) else [formats]
    if "qasm3" in [item.alias for item in formats]:
        device.set_target_program_type("qasm3")
    job = device.run(ghz(n), shots=1024)    # the circuit before transpiling
    job.wait_for_final_state()
    ghz_counts = job.result().data.get_counts()
    both = ghz_counts.get("0" * n, 0) + ghz_counts.get("1" * n, 0)
    print("fraction of shots that are all zeros or all ones:", both / 1024)

#### Your turn 3

The estimate from Part 1 says that a GHZ state on $n$ qubits succeeds with a probability of about $(1 - \varepsilon)^{n-1}$. With $\varepsilon = 0.005$, find the largest $n$ for which the estimate is still at least 0.5, and store it in `n_max`. Let Python do the search: start small and add one qubit at a time. In Python the estimate is `(1 - eps) ** (n - 1)`.

In [ ]:
eps = 0.005
n_max = None
# YOUR CODE HERE
print("largest n with an estimate of at least 0.5:", n_max)

In [ ]:
assert abs(eps - 0.005) < 1e-12, "Keep the line eps = 0.005 as it is. The exercise asks for that value."
assert n_max is not None, "Set n_max. Add one qubit at a time until the estimate drops below one half."
assert not hasattr(n_max, "__len__"), "n_max must be one number, not a list, an array or a string. Keep only the largest n."
assert float(n_max).is_integer(), "n_max is a number of qubits, so it must be a whole number. Add one qubit at a time instead of solving for n."
assert (1 - eps) ** (n_max - 1) >= 0.5, "Too big: the estimate for n_max qubits is already below 0.5."
assert (1 - eps) ** n_max < 0.5, "Too small: one more qubit still keeps the estimate at or above 0.5."
print("Check passed.")
print("estimate for n_max qubits:  ", round((1 - eps) ** (n_max - 1), 4))
print("estimate for one more qubit:", round((1 - eps) ** n_max, 4))

The estimate counts only the two-qubit gates, with one illustrative value of $\varepsilon$. A processor also has readout errors, as the 20-qubit example in Part 1 showed, so expect the fraction you measure to fall below one half at a smaller $n$ than the estimate says. How far you can push it is Challenge 1.

## What you should see

*Slide 44.*

| Lab | Expected result |
|---|---|
| Lab 1: Bell pair, simulator | Only 00 and 11, about 512 each |
| Lab 2: Bell pair, hardware | Mostly 00 and 11, plus a few 01 and 10 from noise |
| Lab 3: GHZ on five qubits, simulator | Only 00000 and 11111 |

## If something breaks

*Slide 45.*

| What you see | What to do |
|---|---|
| The job stays queued | Load the cached result in the next cell and keep going |
| Bitstrings look reversed | Qiskit prints qubit 0 on the right |
| The backend rejects the circuit | Run it through the pass manager and submit the transpiled circuit |
| NameError for sampler or qc | Run the earlier cells in order, top to bottom |
| `AerError: 'unknown instruction: h'` | The circuit was not transpiled for the backend. Run it through the pass manager and submit the transpiled circuit |

The last row is not on the slide. It is the message the simulated processor gives when it rejects a circuit.

## Check yourself

*Slide 40.*

1. A qubit is in $\frac{\sqrt{3}}{2}\lvert 0 \rangle + \frac{1}{2}\lvert 1 \rangle$. How often do you read 1?
2. What does applying H twice do to any qubit?
3. Which outcomes can a Bell pair give, and how often?

## Tonight in one slide

*Slide 46.*

- **A qubit.** Two amplitudes; square them to get probabilities.
- **Gates.** Rotations: X, Z, H, and the two-qubit CX.
- **Entanglement.** Bell and GHZ states: outcomes that always agree.
- **Four steps.** Map, optimize, execute, post-process.

## Challenge 1: Biggest GHZ state

*Slide 47.*

Entangle the most qubits in a GHZ state on real hardware.

Start from tonight's notebook. The challenge notebook is `Qiskit Fall Fest 2026 Challenge 1.ipynb`, in the same folder as this notebook. Submission steps are in the challenge notebook.

## Where to go from here

*Slides 48 and 49.*

- **Tonight.** Finish Lab 3 and start Challenge 1.
- **This week.** Rerun the Bell pair on a different backend and compare the noise.
- **Stuck?** Ask in Discord between sessions.

To choose a backend yourself in Lab 2 or in the hardware cell of Lab 3, print the list with `print(service.backends())` and replace the `least_busy` call by `backend = service.backend("name")`, with a name from that list.

If you ran Lab 2 with `USE_HARDWARE = True`, you ran a circuit on a quantum computer today.

- Notebook, slides, and Challenge 1 are posted on the event page tonight.
- Questions between sessions go in Discord.
- Tomorrow at 5:00 PM: Session 2, quantum algorithms.